In [1]:
import pandas as pd
import xlrd
import openpyxl
from pandas import ExcelWriter
from datetime import datetime
import os
from shutil import copyfile
from openpyxl.utils.cell import get_column_letter
import urllib.request, json
import geopoll_functions
import geopoll_modified
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import re

In [2]:
# --- Basic Survey Metadata ---
template_version = "20250708"
iso3 = "MMR"
round_number = "13"
admin_level = "Admin 2"
user_name = "Edoardo"
language = "EN"

In [3]:
import os
import glob
import polars as pl

# --- 1. Target File to Validate ---
# This is the "Actual" file you just received from the country team
Q_file_path = r"C:\Temp\MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx"

# --- 2. Baseline Configuration ---
Q_baseline_option = "Template" # Options: "Template" or "Custom"
Q_custom_path = r""            # Only used if option is "Custom"

# --- 3. Master Repository Path ---
# This is where all your official FAO/GeoPoll templates live
TEMPLATE_REPO = r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"  

In [4]:

import os
import glob

def get_latest_template(repo_path, enumerator, lang):
    """
    Finds the correct template based on provider (geopoll/kobo) and language.
    """
    # 1. Create a search pattern: e.g., "*geopoll_EN_template*.xlsx"
    # This ignores the version number and ISO3 tag, which change often.
    pattern = f"*_{enumerator.lower()}_{lang.upper()}_template*.xlsx"
    search_path = os.path.join(repo_path, pattern)
    
    # 2. Get all matching files
    matching_files = glob.glob(search_path)
    
    if not matching_files:
        raise FileNotFoundError(f"❌ No template found for {enumerator} in {lang} at {repo_path}")

    # 3. If multiple versions exist (e.g., 20250708 vs 2024...), pick the most recent one
    # Sorting by name usually works for date-coded versions, 
    # but sorting by modification time is safer.
    matching_files.sort(key=os.path.getmtime, reverse=True)
    
    return matching_files[0]

# --- Questionnaire Initialization ---
# We detect these from the file you want to validate
enumerator = "geopoll" # Returns "geopoll"
lang = language      # Returns "EN", "FR", etc.

# --- Step 1: Baseline Selection ---
execution_messages = ""
if Q_baseline_option == "Custom":
    Q_baseline = Q_custom_path
    baseline_source = "Custom File"
else:
    # This automatically finds the right file from your folder screenshot
    Q_baseline = get_latest_template(TEMPLATE_REPO, enumerator, lang)
    baseline_source = f"Master Template ({enumerator} {lang})"

execution_messages += f"\n- Baseline Source: {baseline_source}"
execution_messages += f"\n- Baseline File: {os.path.basename(Q_baseline)}"

print(execution_messages)


- Baseline Source: Master Template (geopoll EN)
- Baseline File: household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx


In [5]:
Q_baseline

'C:\\Users\\edoar\\Food and Agriculture Organization\\OER - 01. DIEM Monitoring\\03. Toolbox\\02. Questionnaires\\household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx'

### Step1 Configuration ###


In [25]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional

ReferenceMode = Literal["latest_template", "custom_reference", "previous_round"]
Enumerator = Literal["geopoll", "kobo"]
Language = Literal["EN", "FR", "ES", "AR", "PT"]

@dataclass
class ValidationConfig:
    template_repo: Path
    working_dir: Path
    questionnaire_file: str
    reference_mode: ReferenceMode
    enumerator: Enumerator
    language: Language
    custom_reference_file: Optional[str] = None
    previous_round_file: Optional[str] = None
    output_subfolder: str = "output"

cfg = ValidationConfig(
    template_repo=Path(r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"),
    working_dir=Path(r"C:\Questionnaire_Validation"),
    questionnaire_file="MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx",
    reference_mode="latest_template",
    enumerator="geopoll",
    language="EN",
    custom_reference_file=None,
    previous_round_file=None,
)

import re
from pathlib import Path

def extract_date_from_name(path: Path) -> int:
    m = re.search(r"template_(\d{8})", path.name)
    return int(m.group(1)) if m else 0

def find_latest_template(template_repo: Path, enumerator: str, language: str) -> Path:
    matches = []
    for path in template_repo.glob("*.xlsx"):
        name = path.name.lower()
        if enumerator.lower() not in name:
            continue
        if f"_{language.lower()}_" not in name:
            continue
        if "template" not in name:
            continue
        matches.append(path)

    if not matches:
        raise FileNotFoundError(
            f"No template found for enumerator={enumerator}, language={language} in {template_repo}"
        )

    matches.sort(key=lambda p: (extract_date_from_name(p), p.stat().st_mtime), reverse=True)
    return matches[0]

def resolve_reference_file(cfg: ValidationConfig) -> Path:
    if cfg.reference_mode == "latest_template":
        return find_latest_template(cfg.template_repo, cfg.enumerator, cfg.language)

    if cfg.reference_mode == "custom_reference":
        if not cfg.custom_reference_file:
            raise ValueError("custom_reference_file is required when reference_mode='custom_reference'")
        return cfg.working_dir / cfg.custom_reference_file

    if cfg.reference_mode == "previous_round":
        if not cfg.previous_round_file:
            raise ValueError("previous_round_file is required when reference_mode='previous_round'")
        return cfg.working_dir / cfg.previous_round_file

    raise ValueError(f"Unsupported reference_mode: {cfg.reference_mode}")

In [26]:
def prepare_run(cfg: ValidationConfig) -> dict:
    cfg.working_dir.mkdir(parents=True, exist_ok=True)
    output_dir = cfg.working_dir / cfg.output_subfolder
    output_dir.mkdir(parents=True, exist_ok=True)

    questionnaire_path = cfg.working_dir / cfg.questionnaire_file
    reference_path = resolve_reference_file(cfg)

    if not questionnaire_path.exists():
        raise FileNotFoundError(f"Questionnaire file not found: {questionnaire_path}")
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference file not found: {reference_path}")

    result_file = output_dir / f"{questionnaire_path.stem.split()[0]}_{cfg.enumerator}_validated.xlsx"

    run_info = {
        "questionnaire_path": questionnaire_path,
        "reference_path": reference_path,
        "result_file": result_file,
    }

    print("Questionnaire:", questionnaire_path)
    print("Reference:    ", reference_path)
    print("Output:       ", result_file)

    return run_info

run = prepare_run(cfg)

Questionnaire: C:\Questionnaire_Validation\MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx
Reference:     C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires\household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
Output:        C:\Questionnaire_Validation\output\MMR_R13_geopoll_validated.xlsx


### step2 normalization ### 

In [35]:
import openpyxl
import polars as pl

import openpyxl
import polars as pl
from pathlib import Path

import openpyxl
import polars as pl
from pathlib import Path

def read_survey_sheet(path: Path, sheet_name: str = "survey", header_row: int = 3) -> pl.DataFrame:
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)

    def get_sheet_case_insensitive(wb, target_name):
        for name in wb.sheetnames:
            if name.strip().lower() == target_name.lower():
                return wb[name]
        raise KeyError(f"Worksheet '{target_name}' not found. Available sheets: {wb.sheetnames}")

    ws = get_sheet_case_insensitive(wb, sheet_name)

    def clean_header(h):
        if h is None:
            return None
        return str(h).replace("\n", " ").strip()

    row_iter = ws.iter_rows(values_only=True)

    for _ in range(header_row - 1):
        next(row_iter)

    raw_headers = next(row_iter)

    headers = [
        clean_header(h) if h is not None else f"unnamed_{i}"
        for i, h in enumerate(raw_headers, start=1)
    ]

    rows = []
    excel_row = header_row + 1

    for values in row_iter:
        if all(v is None for v in values):
            excel_row += 1
            continue

        row_dict = {headers[i]: values[i] for i in range(len(headers))}
        row_dict["excel_row"] = excel_row
        row_dict["source_file"] = path.name
        rows.append(row_dict)

        excel_row += 1

    df = pl.DataFrame(rows)

    if "Q Name" not in df.columns:
        raise ValueError(f"'Q Name' column not found in {path.name}")

    df = df.filter(pl.col("Q Name").is_not_null())

    df = df.with_columns(
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().alias("Q Name")
    )

    return df

CORE_COLUMNS = [
    "Q Name",
    "Suggested Qname",
    "English",
    "French",
    "Spanish",
    "Arabic",
    "Portuguese",
    "Q Type",
    "Mandatory",
    "Default skip patterns & conditional",
    "Specific skip pattern variable (from blue text)",
    "Additional notes",
    "excel_row",
    "source_file",
]

def build_questions_df(df: pl.DataFrame) -> pl.DataFrame:
    available = [c for c in CORE_COLUMNS if c in df.columns]
    out = df.select(available)

    for c in out.columns:
        if c not in ["excel_row"]:
            out = out.with_columns(
                pl.col(c).cast(pl.Utf8).fill_null("").str.strip_chars().alias(c)
            )

    return out
import re

OPTION_PATTERN = re.compile(r"(?ms)^\s*(\d+)\)\s*(.*?)(?=^\s*\d+\)|\Z)")

def parse_option_blocks(text: str):
    if not text:
        return []
    return [(int(code), label.strip()) for code, label in OPTION_PATTERN.findall(text)]

def explode_options(questions_df: pl.DataFrame, text_col: str = "English") -> pl.DataFrame:
    rows = []

    needed = [c for c in ["Q Name", "Q Type", "excel_row", text_col] if c in questions_df.columns]
    for row in questions_df.select(needed).iter_rows(named=True):
        q_name = row.get("Q Name", "")
        q_type = (row.get("Q Type", "") or "").strip()
        excel_row = row.get("excel_row")
        text = row.get(text_col, "")

        if q_type not in {"Single Choice", "Select All That Apply"}:
            continue

        parsed = parse_option_blocks(text)
        for option_code, option_label in parsed:
            rows.append(
                {
                    "Q Name": q_name,
                    "Q Type": q_type,
                    "option_code": option_code,
                    "option_label": option_label,
                    "text_column": text_col,
                    "excel_row": excel_row,
                }
            )

    if not rows:
        return pl.DataFrame(
            schema={
                "Q Name": pl.Utf8,
                "Q Type": pl.Utf8,
                "option_code": pl.Int64,
                "option_label": pl.Utf8,
                "text_column": pl.Utf8,
                "excel_row": pl.Int64,
            }
        )

    return pl.DataFrame(rows)

In [36]:
current_raw = read_survey_sheet(run["questionnaire_path"])
reference_raw = read_survey_sheet(run["reference_path"])

current_questions = build_questions_df(current_raw)
reference_questions = build_questions_df(reference_raw)

current_options_en = explode_options(current_questions, text_col="English")
reference_options_en = explode_options(reference_questions, text_col="English")

In [ ]:
print(current_questions.shape)
print(reference_questions.shape)
print(current_options_en.shape)
print(current_options_en.head())

(207, 8)
(180, 8)
(1259, 6)
shape: (5, 6)
┌───────────┬───────────────┬─────────────┬────────────────────┬─────────────┬───────────┐
│ Q Name    ┆ Q Type        ┆ option_code ┆ option_label       ┆ text_column ┆ excel_row │
│ ---       ┆ ---           ┆ ---         ┆ ---                ┆ ---         ┆ ---       │
│ str       ┆ str           ┆ i64         ┆ str                ┆ str         ┆ i64       │
╞═══════════╪═══════════════╪═════════════╪════════════════════╪═════════════╪═══════════╡
│ calldispo ┆ Single Choice ┆ 1           ┆ Someone answers    ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 2           ┆ No answer          ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 3           ┆ Refusal            ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 4           ┆ Call back          ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 5           ┆ Non-Working Number ┆ English     ┆ 6         │
└───────────┴───────────────┴─────────────┴─────

### Mismatch ###

In [53]:
def normalize_text(col):
    return (
        pl.col(col)
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")   # remove punctuation
        .str.replace_all(r"\s+", " ")      # normalize spaces
        .str.strip_chars()
    )

def normalize_mandatory(col):
    return (
        pl.col(col)
        .cast(pl.Utf8)
        .str.to_lowercase()
        .str.strip_chars()
        .map_elements(lambda x: "yes" if x in ["yes", "y", "true", "1"]
                      else "no" if x in ["no", "n", "false", "0"]
                      else "")
    )

In [ ]:
current_q = current_questions.select("Q Name").unique()
reference_q = reference_questions.select("Q Name").unique()

added = current_q.join(reference_q, on="Q Name", how="anti").sort("Q Name")
removed = reference_q.join(current_q, on="Q Name", how="anti").sort("Q Name")

print("Added questions:", added.height)
print(added)

print("Removed questions:", removed.height)
print(removed)


mandatory_diff = (
    current_questions
    .select(["Q Name", "Mandatory"])
    .join(
        reference_questions.select(["Q Name", "Mandatory"]),
        on="Q Name",
        how="inner",
        suffix="_ref"
    )
    .with_columns([
        normalize_mandatory("Mandatory").alias("Mandatory_norm"),
        normalize_mandatory("Mandatory_ref").alias("Mandatory_ref_norm")
    ])
    .filter(pl.col("Mandatory_norm") != pl.col("Mandatory_ref_norm"))
    .sort("Q Name")
)

print("Mandatory mismatches:", mandatory_diff.height)
print(mandatory_diff)
option_diff = (
    current_options_en
    .join(
        reference_options_en,
        on=["Q Name", "option_code"],
        how="inner",
        suffix="_ref"
    )
    # 👉 create normalized columns
    .with_columns([
        normalize_text("option_label").alias("option_label_norm"),
        normalize_text("option_label_ref").alias("option_label_ref_norm")
    ])
    # 👉 compare normalized values
    .filter(pl.col("option_label_norm") != pl.col("option_label_ref_norm"))
    .sort(["Q Name", "option_code"])
)
print("Option label mismatches:", option_diff.height)
print(option_diff.head(20))

Added questions: 97
shape: (97, 1)
┌───────────────┐
│ Q Name        │
│ ---           │
│ str           │
╞═══════════════╡
│ FCSDairy      │
│ FCSFat        │
│ FCSFruit      │
│ FCSPr         │
│ FCSPulse      │
│ …             │
│ rCSILessQlty  │
│ rCSIMealAdult │
│ rCSIMealNb    │
│ rCSIMealSize  │
│ rcsi          │
└───────────────┘
Removed questions: 70
shape: (70, 1)
┌──────────────────────────────┐
│ Q Name                       │
│ ---                          │
│ str                          │
╞══════════════════════════════╡
│ contact_2                    │
│ contact_2_name               │
│ contact_3                    │
│ contact_3_name               │
│ covid                        │
│ …                            │
│ rcsi_limit_portions          │
│ rcsi_reduce_number_meals     │
│ rcsi_restrict_adult_consumpt │
│ resp_firstname               │
│ resp_lastname                │
└──────────────────────────────┘


### mismatch2 ###

In [57]:
import polars as pl


# =========================
# Normalization helpers
# =========================

def normalize_text_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")   # remove punctuation
        .str.replace_all(r"\s+", " ")      # normalize spaces
        .str.strip_chars()
    )


def normalize_mandatory_expr(col_name: str) -> pl.Expr:
    cleaned = (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.strip_chars()
    )

    return (
        pl.when(cleaned.is_in(["yes", "y", "true", "1"]))
        .then(pl.lit("yes"))
        .when(cleaned.is_in(["no", "n", "false", "0"]))
        .then(pl.lit("no"))
        .otherwise(pl.lit(""))
    )


# =========================
# Generic comparison helpers
# =========================

def compare_question_presence(
    current_questions: pl.DataFrame,
    reference_questions: pl.DataFrame
) -> tuple[pl.DataFrame, pl.DataFrame]:
    current_q = current_questions.select("Q Name").unique()
    reference_q = reference_questions.select("Q Name").unique()

    added = current_q.join(reference_q, on="Q Name", how="anti").sort("Q Name")
    removed = reference_q.join(current_q, on="Q Name", how="anti").sort("Q Name")

    return added, removed


def compare_mandatory(
    current_questions: pl.DataFrame,
    reference_questions: pl.DataFrame,
    treat_blank_as_no: bool = False
) -> pl.DataFrame:
    df = (
        current_questions
        .select(["Q Name", "Mandatory", "excel_row"])
        .join(
            reference_questions.select(["Q Name", "Mandatory", "excel_row"]),
            on="Q Name",
            how="inner",
            suffix="_ref"
        )
        .with_columns([
            normalize_mandatory_expr("Mandatory").alias("Mandatory_norm"),
            normalize_mandatory_expr("Mandatory_ref").alias("Mandatory_ref_norm")
        ])
    )

    if treat_blank_as_no:
        mismatch_filter = (
            (pl.col("Mandatory_norm") != pl.col("Mandatory_ref_norm")) &
            ~(
                (pl.col("Mandatory_norm") == "") &
                (pl.col("Mandatory_ref_norm") == "no")
            ) &
            ~(
                (pl.col("Mandatory_norm") == "no") &
                (pl.col("Mandatory_ref_norm") == "")
            )
        )
    else:
        mismatch_filter = pl.col("Mandatory_norm") != pl.col("Mandatory_ref_norm")

    return (
        df
        .filter(mismatch_filter)
        .with_columns([
            pl.lit("mandatory_mismatch").alias("issue_type"),
            pl.lit("Mandatory").alias("field")
        ])
        .sort("Q Name")
    )


def compare_option_labels(
    current_options: pl.DataFrame,
    reference_options: pl.DataFrame
) -> pl.DataFrame:
    return (
        current_options
        .join(
            reference_options,
            on=["Q Name", "option_code"],
            how="inner",
            suffix="_ref"
        )
        .with_columns([
            normalize_text_expr("option_label").alias("option_label_norm"),
            normalize_text_expr("option_label_ref").alias("option_label_ref_norm")
        ])
        .filter(pl.col("option_label_norm") != pl.col("option_label_ref_norm"))
        .with_columns([
            pl.lit("option_label_mismatch").alias("issue_type"),
            pl.lit("option_label").alias("field")
        ])
        .sort(["Q Name", "option_code"])
    )


# =========================
# Critical sets
# =========================

CRITICAL_SETS = {
    "HDDS": [
        {"q_name": "hdds", "expected_mandatory": "yes", "required": True},
        {"q_name": "hdds_confirmation", "expected_mandatory": "yes", "required": True},
    ],
    "WEALTH": [
        {"q_name": "hh_wealth", "expected_mandatory": "yes", "required": True},
    ],
    "RCSI": [
        {"q_name": "rCSIlessQlty", "expected_mandatory": "yes", "required": True},
        {"q_name": "rCSIMealAdult", "expected_mandatory": "yes", "required": True},
        {"q_name": "rCSIMealNb", "expected_mandatory": "yes", "required": True},
        {"q_name": "rCSIMealSize", "expected_mandatory": "yes", "required": True},
        {"q_name": "rcsi", "expected_mandatory": "yes", "required": True},
    ],
    # Add crop harvest carefully later, because some questions may be conditional
}


def validate_critical_sets(
    questions_df: pl.DataFrame,
    critical_sets: dict
) -> pl.DataFrame:
    # Make sure normalized mandatory exists
    if "Mandatory_norm" not in questions_df.columns:
        questions_df = questions_df.with_columns(
            normalize_mandatory_expr("Mandatory").alias("Mandatory_norm")
        )

    issues = []
    present_qnames = set(questions_df["Q Name"].to_list())

    for set_name, rules in critical_sets.items():
        present_in_set = [r["q_name"] for r in rules if r["q_name"] in present_qnames]

        # partial presence check
        if 0 < len(present_in_set) < len(rules):
            issues.append({
                "issue_type": "partial_critical_set",
                "set_name": set_name,
                "Q Name": "",
                "field": "Q Name",
                "current": ", ".join(sorted(present_in_set)),
                "reference": f"Expected {len(rules)} questions in set",
                "severity": "high",
                "excel_row": None,
            })

        for rule in rules:
            q_name = rule["q_name"]
            expected = rule.get("expected_mandatory", "")
            required = rule.get("required", True)

            if q_name not in present_qnames:
                if required:
                    issues.append({
                        "issue_type": "missing_critical_question",
                        "set_name": set_name,
                        "Q Name": q_name,
                        "field": "Q Name",
                        "current": "",
                        "reference": "present",
                        "severity": "high",
                        "excel_row": None,
                    })
                continue

            row = (
                questions_df
                .filter(pl.col("Q Name") == q_name)
                .select(["Q Name", "Mandatory", "Mandatory_norm", "excel_row"])
                .to_dicts()
            )

            if not row:
                continue

            row = row[0]

            if expected and row["Mandatory_norm"] != expected:
                issues.append({
                    "issue_type": "critical_mandatory_mismatch",
                    "set_name": set_name,
                    "Q Name": q_name,
                    "field": "Mandatory",
                    "current": row["Mandatory"],
                    "reference": expected,
                    "severity": "high",
                    "excel_row": row.get("excel_row"),
                })

    if not issues:
        return pl.DataFrame(
            schema={
                "issue_type": pl.Utf8,
                "set_name": pl.Utf8,
                "Q Name": pl.Utf8,
                "field": pl.Utf8,
                "current": pl.Utf8,
                "reference": pl.Utf8,
                "severity": pl.Utf8,
                "excel_row": pl.Int64,
            }
        )

    return pl.DataFrame(issues)


# =========================
# Optional: unify issues
# =========================

def make_presence_issues(added: pl.DataFrame, removed: pl.DataFrame) -> pl.DataFrame:
    added_issues = (
        added
        .with_columns([
            pl.lit("added_question").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("Q Name").alias("field"),
            pl.lit("present").alias("current"),
            pl.lit("missing_in_reference").alias("reference"),
            pl.lit("medium").alias("severity"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )

    removed_issues = (
        removed
        .with_columns([
            pl.lit("removed_question").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("Q Name").alias("field"),
            pl.lit("missing_in_current").alias("current"),
            pl.lit("present").alias("reference"),
            pl.lit("high").alias("severity"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )

    return pl.concat([added_issues, removed_issues], how="vertical")


def make_mandatory_issues(mandatory_diff: pl.DataFrame) -> pl.DataFrame:
    return (
        mandatory_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("Mandatory").alias("current"),
            pl.col("Mandatory_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_issues(option_diff: pl.DataFrame) -> pl.DataFrame:
    return (
        option_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
            pl.col("excel_row"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


# =========================
# Main pipeline
# =========================

# current_raw = read_survey_sheet(run["questionnaire_path"])
# reference_raw = read_survey_sheet(run["reference_path"])

# current_questions = build_questions_df(current_raw)
# reference_questions = build_questions_df(reference_raw)

# current_options_en = explode_options(current_questions, text_col="English")
# reference_options_en = explode_options(reference_questions, text_col="English")

# Compare
added, removed = compare_question_presence(current_questions, reference_questions)
mandatory_diff = compare_mandatory(current_questions, reference_questions, treat_blank_as_no=False)
option_diff = compare_option_labels(current_options_en, reference_options_en)

# Critical rules
critical_issues = validate_critical_sets(current_questions, CRITICAL_SETS)

# Unified issues table
presence_issues = make_presence_issues(added, removed)
mandatory_issues = make_mandatory_issues(mandatory_diff)
option_issues = make_option_issues(option_diff)

all_issues = pl.concat(
    [presence_issues, mandatory_issues, option_issues, critical_issues],
    how="vertical"
).sort(["severity", "issue_type", "Q Name"])

print("Added questions:", added.height)
print("Removed questions:", removed.height)
print("Mandatory mismatches:", mandatory_diff.height)
print("Option label mismatches:", option_diff.height)
print("Critical issues:", critical_issues.height)
print("All issues:", all_issues.height)

print(all_issues.head(20))

Added questions: 97
Removed questions: 70
Mandatory mismatches: 14
Option label mismatches: 14
Critical issues: 9
All issues: 204
shape: (20, 8)
┌───────────────┬──────────┬──────────────┬───────────┬─────────┬───────────┬──────────┬───────────┐
│ issue_type    ┆ set_name ┆ Q Name       ┆ field     ┆ current ┆ reference ┆ severity ┆ excel_row │
│ ---           ┆ ---      ┆ ---          ┆ ---       ┆ ---     ┆ ---       ┆ ---      ┆ ---       │
│ str           ┆ str      ┆ str          ┆ str       ┆ str     ┆ str       ┆ str      ┆ i64       │
╞═══════════════╪══════════╪══════════════╪═══════════╪═════════╪═══════════╪══════════╪═══════════╡
│ critical_mand ┆ RCSI     ┆ rCSIMealAdul ┆ Mandatory ┆ no      ┆ yes       ┆ high     ┆ 200       │
│ atory_mismatc ┆          ┆ t            ┆           ┆         ┆           ┆          ┆           │
│ h             ┆          ┆              ┆           ┆         ┆           ┆          ┆           │
│ critical_mand ┆ RCSI     ┆ rCSIMealNb   ┆ Man